# Machine Learning Pipeline

Now that we have experience preparing data for input to machine learning libraries, the next step will be to train, tune, and test a model.  You will perform all three of these steps in this hands-on activity.

The assignment consists of the following steps:

1. Load two datasets and prepare their representations and labels for model input. 
2. Split the data into training and testing.
3. Select a model, and identify the parameters to tune.
4. Tune the model.
5. Evaluate the model's performance.

In [2]:
import logging
logging.getLogger("scapy.runtime").setLevel(logging.ERROR)

from netml.pparser.parser import PCAP
from netml.utils.tool import dump_data, load_data

import pandas as pd

## Convert the Packet Capture Into Flows

1. Load the two packet captures for HTTP requests and Log4j scan, 
2. convert them into traffic flows, 
3. generate features from the flow,  
4. label the traffic,
5. normalize your labeled features into a 2D matrix

In [3]:
hpcap = PCAP('data/http.pcap', flow_ptks_thres=2, verbose=10)
lpcap = PCAP('data/log4j.pcap', flow_ptks_thres=2, verbose=10)

# Convert the packet captures into flows
hflows = hpcap.pcap2flows()
lflows = lpcap.pcap2flows()

lpcap.flow2features('IAT', fft=False, header=True)
ld = pd.DataFrame(lpcap.features)

hpcap.flow2features('IAT', fft=False, header=True)
hd = pd.DataFrame(hpcap.features)

pcap_file: data/http.pcap
ith_packets: 0
ith_packets: 10000
ith_packets: 20000
len(flows): 593
total number of flows: 593. Num of flows < 2 pkts: 300, and >=2 pkts: 293 without timeout splitting.
kept flows: 293. Each of them has at least 2 pkts after timeout splitting.
flow_durations.shape: (293, 1)
        col_0
count 293.000
mean   11.629
std    15.820
min     0.000
25%     0.076
50%     0.455
75%    20.097
max    46.235
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 293 entries, 0 to 292
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   col_0   293 non-null    float64
dtypes: float64(1)
memory usage: 2.4 KB
None
0th_flow: len(pkts): 4
After splitting flows, the number of subflows: 291 and each of them has at least 2 packets.
pcap_file: data/log4j.pcap
ith_packets: 0
ith_packets: 10000
ith_packets: 20000
ith_packets: 30000
ith_packets: 40000
ith_packets: 50000
ith_packets: 60000
ith_packets: 70000
ith_packets: 80000
len

In [4]:
hds = hd.loc[:, :4]
pd.set_option('mode.chained_assignment', None)
hds['label'] = 0
ld['label'] = 1

data = pd.concat([ld, hds])
X = data.loc[:, :4]
y = data['label']

print(X.shape, y.shape)
print(y.value_counts())

data.head(10)


(5086, 5) (5086,)
label
1    4795
0     291
Name: count, dtype: int64


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,label
0,1.000,1.000,0.000,1.000,5.000,0.000,0.000,0.000,52.000,52.000,52.000,52.000,52.000,0.012,0.000,0.013,5.041,0.097,1
1,1.000,1.000,0.000,1.000,5.000,0.000,0.000,0.000,64.000,64.000,64.000,64.000,64.000,0.012,0.001,5.004,0.146,0.000,1
2,0.000,4.000,0.000,0.000,0.000,0.000,0.000,0.000,54.000,54.000,54.000,54.000,0.000,1.001,2.020,4.253,0.000,0.000,1
3,0.000,10.000,0.000,0.000,0.000,0.000,0.000,0.000,243.000,243.000,243.000,243.000,243.000,1.816,1.001,1.001,1.001,1.000,1
4,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,244.000,244.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1
5,0.000,1.000,1.000,1.000,3.000,0.000,1.000,1.000,120.000,120.000,120.000,120.000,0.000,0.176,0.000,0.177,0.000,0.000,1
6,0.000,1.000,0.000,1.000,3.000,0.000,0.000,0.000,64.000,64.000,64.000,0.000,0.000,0.176,0.001,0.000,0.000,0.000,1
7,0.000,10.000,0.000,0.000,0.000,0.000,0.000,0.000,244.000,244.000,244.000,244.000,244.000,1.925,1.000,1.001,1.001,1.000,1
8,1.000,1.000,0.000,1.000,5.000,0.000,0.000,0.000,54.000,54.000,54.000,54.000,54.000,0.144,0.001,0.145,0.002,0.144,1
9,1.000,1.000,0.000,1.000,4.000,0.000,0.000,0.000,64.000,64.000,64.000,64.000,0.000,0.145,0.001,0.146,0.000,0.000,1


## Evaluating a Machine Learning Model

The goal of supervised learning is to train a model that takes examples and predicts labels for these examples that are as close as possible to the actual labels. For instance, in this example above, a model might take features from a traffic trace and predict whether the traffic constitutes regular web traffic or a scan.

How do you measure whether the model is succeeding if you don't know the true labels for new observations? The way to solve this problem is to test the performance of the trained algorithm on additional data that it has never seen, but for which you already know the correct labels. 

This requires that you train the algorithm using only a portion of the entire labeled dataset (the **training set**) and withold the rest of the labeled data (the **test set**) for testing how well the model generalizes to new information. 

To evaluate the model, we will need to split the data into train and test sets.

### Split into Training and Test Sets

Split your data into a training and test set using scikit-learn. A common split is to train on 80% of your data, while withholding 20% of the data. 

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size = 0.2, # 20% test, 80% train
    random_state = 42, #reproducibile split 
    stratify = y # ensure the same split for each run

)

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

print(y_train.value_counts())
print(y_test.value_counts())


X_test.head(10)

Train: (4068, 5) (4068,)
Test: (1018, 5) (1018,)
label
1    3835
0     233
Name: count, dtype: int64
label
1    960
0     58
Name: count, dtype: int64


,0,1,2,3,4
4142,0.000,3.000,0.000,0.000,0.000
4735,0.000,10.000,0.000,0.000,0.000
2039,1.000,1.000,0.000,1.000,5.000
252,0.000,1.000,0.000,5.000,15.000
2768,1.000,1.000,0.000,1.000,4.000
3847,1.000,1.000,0.000,1.000,5.000
3118,0.000,2.000,0.000,0.000,0.000
2868,0.000,4.000,0.000,0.000,0.000
723,1.000,1.000,0.000,1.000,4.000
2219,1.000,1.000,0.000,1.000,6.000


### Training Your Model

Now that you have split your data into training and testing sets, you are ready to train and evaluate a model. 

Import a machine learning model of your choice, use your training set to train the model, and use the test set to evaluate it. 

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report 

model = RandomForestClassifier(
    random_state = 42, 
    n_jobs = -1, #use all CPU cores
    n_estimators = 100, #number of trees
    class_weight = 'balanced' #weight classes by inverse of their frequency
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))



Accuracy: 0.9842829076620825
              precision    recall  f1-score   support

           0       0.79      0.98      0.88        58
           1       1.00      0.98      0.99       960

    accuracy                           0.98      1018
   macro avg       0.90      0.98      0.93      1018
weighted avg       0.99      0.98      0.99      1018



In [7]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_grid, 
    cv = 5, 
    scoring = 'f1', 
    n_jobs = -1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)





Best Parameters: {'max_depth': None, 'n_estimators': 200}
Best Score: 0.9906752498569791


### Test Your Trained Model

You can now evaluate how well your trained model works.  There are several valuable ways to visualize your results. You might use techniques such as a confusion matrix, or a receiver operating characteristic (ROC) curve. Below we will gain some experience plotting both of those.  This [documentation](https://scikit-learn.org/stable/auto_examples/miscellaneous/plot_display_object_visualization.html) may help you with plotting these results.

#### Confusion Matrix 

A confusion matrix is a one way to understand errors of different types. We can see a lot of examples off diagonal, suggesting a fair number of incorrect answers.

In [8]:
print((y_test ==1).mean()) 
print(X_train[y_train == 0].describe())
print(X_train[y_train == 1].describe())

from sklearn.metrics import confusion_matrix 
print(confusion_matrix(y_test, y_pred))

0.9430255402750491
            0       1       2       3        4
count 233.000 233.000 233.000 233.000  233.000
mean    0.236   0.605   0.077  12.824   89.253
std     0.525   0.499   0.375  31.534  372.009
min     0.000   0.000   0.000   0.000    0.000
25%     0.000   0.000   0.000   2.000    4.000
50%     0.000   1.000   0.000   4.000   10.000
75%     0.000   1.000   0.000   7.000   15.000
max     2.000   2.000   3.000 221.000 2985.000
             0        1        2        3        4
count 3835.000 3835.000 3835.000 3835.000 3835.000
mean     0.629    2.052    0.145    0.867    3.316
std      0.558    7.898    0.488    2.166    3.798
min      0.000    0.000    0.000    0.000    0.000
25%      0.000    1.000    0.000    0.000    0.000
50%      1.000    1.000    0.000    1.000    4.000
75%      1.000    2.000    0.000    1.000    5.000
max      7.000  477.000    4.000   37.000   75.000
[[ 57   1]
 [ 15 945]]


#### Receiver Operating Characteristic

Some models can output different classes based on a threshold that is set for the decision. 

#### Area Under the Curve (AUC)

From the ROC above, you can also compute a metric called the area under the curve (AUC). Visually, this is the area under the curve that you just plotted. You could see, intuitively, that the "best" performance should yield an AUC of 1, and the worst performance would yield an AUC closer to 0.5.

Scikit learn also has a function for computing AUC.  Compute the area under the curve.

In [9]:
from sklearn.metrics import roc_auc_score 

y_score = model.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, y_score)
print("AUC Score:", auc_score)





AUC Score: 0.990014367816092


## Thought Question

Which evaluation model is more appropriate, and when (i.e., under what circumstances)? When might you care more about looking at the confusion matrix (or model accuracy) vs. the ROC, or the area under the curve?

confusion matrix gives a holinstic overview with summary analysis 
ROC and the other values give more specific relational percentages accounting for relative accuracy levels 